# FockPARFLM v2 — TinyStories Diagnostic & Tuning Notebook

## Purpose

Debug and performance-tune the Q/K/V-structured Fock mechanism on a **small TinyStories subset**
before committing to a full-scale P10g run. The F2 Dyck₂ falsifier showed a clean +5.37 pp
expressivity lift (49.01% deep-test acc vs 43.64% baseline), but there are structural reasons
why this advantage may not transfer fully to natural language perplexity.

## Hypothesised bottlenecks

| ID | Bottleneck | Mechanism | Diagnostic |
|---|---|---|---|
| B1 | **Register compression** | M=16 slots over T=512 positions → 97% information discarded | Register attention entropy, per-register coverage |
| B2 | **Shared creation gate** | Same W_K, W_V across all L layers → can't extract layer-specific features | Per-layer creation attention similarity |
| B3 | **Salience blending stale** | `blend = salience ≈ 1` at deep layers → creation output ignored | Track blend ratio, register content cosine similarity across layers |
| B4 | **No specialised register-token path** | Registers enter PARF dynamics as additional particles → interaction through generic V_phi/V_theta | Measure register-token force magnitude vs token-token |
| B5 | **Reverse channel under-used** | `tanh(scale)` starts at 0; may stay near 0 if gradient signal is weak | Track reverse_channel_scale value, gradient magnitude |
| B6 | **V_theta capacity ceiling** | PARFLM P10h saturated at 26.4 PPL; Fock v2 inherits this limit | Compare FockPARF v2 vs PARFLM baseline on identical configs |

## Cell configurations

| Cell | Config | Purpose |
|---|---|---|
| `D1` | FockPARFLM v2 (P10g recipe + v2 registers), M=16 | Baseline diagnostic — matches Dyck₂ F2 register count at TinyStories scale |
| `D2` | Same as D1 but M=32 | Doubles register capacity |
| `D3` | Same as D1 but per-layer creation gates (not shared) | Tests B2 |
| `D4` | Same as D1 but `blend = 0.5 * salience` (halved persistence) | Tests B3 |
| `D5` | PARFLM baseline (no registers, P10g recipe) | Control arm for B6 |
| `D6` | Same as D1 but `tau_create_init=0.1` (learnable temperature) | Tests B1 fix — sharpen creation attention with learnable τ instead of 1/√d_k |

**Scale**: d=256, L=8, T=256 (half-length for speed), batch=8, 2000 steps on 1M tokens.
Expected wall-clock: ~20-40 min per arm on a T4/A100 GPU.

## Run protocol

Run the notebook once per `CELL` value. All arms use the same data split,
hyperparameters, and evaluation protocol. The diagnostic section (§8) runs
automatically and logs detailed per-layer register statistics.

## 0. Environment setup + cell selector

In [ ]:
CELL = 'D1'   # one of: 'D1' | 'D2' | 'D3' | 'D4' | 'D5' | 'D6'
SEED = 0

REPO_URL        = 'https://github.com/dimitarpg13/semsimula.git'
REPO_BRANCH     = 'main'
COLAB_REPO_PATH = '/content/semsimula'
GDRIVE_OUT_REL  = 'semsimula_fock_v2_debug'

import os, sys, shutil, subprocess, json, time, math
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd: str) -> None:
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'command failed (exit {r.returncode}): {cmd}')


if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    GDRIVE_OUT = Path('/content/drive/MyDrive') / GDRIVE_OUT_REL
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print(f'GDrive output root = {GDRIVE_OUT}')

    REPO_ROOT = Path(COLAB_REPO_PATH)
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    _sh(f'git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {COLAB_REPO_PATH}')

    DATA_CACHE = GDRIVE_OUT / 'data'
    DATA_CACHE.mkdir(exist_ok=True)
    repo_data_dir = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    if repo_data_dir.is_symlink():
        repo_data_dir.unlink()
    elif repo_data_dir.is_dir():
        shutil.rmtree(repo_data_dir)
    repo_data_dir.symlink_to(DATA_CACHE)
    print(f'data/ -> {DATA_CACHE}')

    RESULTS_ROOT = GDRIVE_OUT
    _sh('pip install -q transformers huggingface_hub pyarrow')
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    RESULTS_ROOT = (
        REPO_ROOT / 'notebooks' / 'conservative_arch' / 'parf'
        / 'results' / 'fock_v2_debug'
    )
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

PARF_DIR    = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'parf'
SCALEUP_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup'
DATA_DIR    = REPO_ROOT / 'notebooks' / 'conservative_arch'
for p in (str(REPO_ROOT), str(DATA_DIR), str(SCALEUP_DIR), str(PARF_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

RUN_DIR = RESULTS_ROOT / CELL / f'seed{SEED}'
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(f'REPO_ROOT    = {REPO_ROOT}')
print(f'RESULTS_ROOT = {RESULTS_ROOT}')
print(f'RUN_DIR      = {RUN_DIR}')
print(f'CELL = {CELL!r}  SEED = {SEED}')

## 1. Device + reproducibility

In [ ]:
import torch
import numpy as np

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')

torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

if torch.cuda.is_available():
    device = 'cuda'
    torch.cuda.manual_seed_all(SEED)
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  ({props.total_memory / 1e9:.1f} GB)')
elif torch.backends.mps.is_available():
    device = 'mps'
    print('Using MPS (Apple Silicon)')
else:
    device = 'cpu'
    print('WARNING: CPU only — will be slow')
print(f'device = {device}')

## 2. Experiment recipes

All arms use the P10g-equivalent architecture as the base (full P5+P7+P8 stack,
`v_hidden=2048`, structural_competitive V_phi), but at reduced scale for fast
iteration: T=256 (half), 1M training tokens, 2000 steps.

In [ ]:
# ── Shared configuration (P10g-scale architecture at debug budget) ──
VOCAB_SIZE     = 50257
D              = 256
L              = 8
MAX_LEN        = 256       # half-length for speed
BLOCK_SIZE     = 256
V_HIDDEN       = 2048
V_DEPTH        = 3
TOP_K          = 4
SCORE_HEAD     = 32
BATCH_SIZE     = 8
TOTAL_STEPS    = 2000
LR             = 3e-4
WEIGHT_DECAY   = 0.01
WARMUP_STEPS   = 100
GRAD_CLIP      = 1.0
EVAL_EVERY     = 200
EVAL_ITERS     = 20
LOG_EVERY      = 50
MAX_TRAIN_TOK  = 1_000_000

# ── Per-cell Fock v2 knobs ──
RECIPES = {
    'D1': {
        'desc': 'FockPARF v2, M=16 (baseline diagnostic)',
        'use_fock_v2': True,
        'n_registers': 16,
        'd_k': 64,
        'register_salience_decay': 0.5,
        'register_salience_threshold': 0.005,
        'destruction_gate_hidden': 64,
        'reverse_channel': True,
        'per_layer_creation': False,
        'blend_scale': 1.0,          # full salience blending (default)
    },
    'D2': {
        'desc': 'FockPARF v2, M=32 (double registers)',
        'use_fock_v2': True,
        'n_registers': 32,
        'd_k': 64,
        'register_salience_decay': 0.5,
        'register_salience_threshold': 0.005,
        'destruction_gate_hidden': 64,
        'reverse_channel': True,
        'per_layer_creation': False,
        'blend_scale': 1.0,
    },
    'D3': {
        'desc': 'FockPARF v2, M=16, d_k=128 (wider queries)',
        'use_fock_v2': True,
        'n_registers': 16,
        'd_k': 128,
        'register_salience_decay': 0.5,
        'register_salience_threshold': 0.005,
        'destruction_gate_hidden': 64,
        'reverse_channel': True,
        'per_layer_creation': False,
        'blend_scale': 1.0,
    },
    'D4': {
        'desc': 'FockPARF v2, M=16, halved persistence (blend_scale=0.5)',
        'use_fock_v2': True,
        'n_registers': 16,
        'd_k': 64,
        'register_salience_decay': 0.5,
        'register_salience_threshold': 0.005,
        'destruction_gate_hidden': 64,
        'reverse_channel': True,
        'per_layer_creation': False,
        'blend_scale': 0.5,          # reduces staleness: blend = 0.5 * salience
    },
    'D5': {
        'desc': 'PARFLM baseline (no registers, P10g recipe)',
        'use_fock_v2': False,
        'n_registers': 0,
    },
    'D6': {
        'desc': 'FockPARF v2, M=16, tau_create=0.1 (learnable temperature)',
        'use_fock_v2': True,
        'n_registers': 16,
        'd_k': 64,
        'register_salience_decay': 0.5,
        'register_salience_threshold': 0.005,
        'destruction_gate_hidden': 64,
        'reverse_channel': True,
        'per_layer_creation': False,
        'blend_scale': 1.0,
        'tau_create_init': 0.1,
    },
}

if CELL not in RECIPES:
    raise ValueError(f'CELL must be one of {list(RECIPES)}; got {CELL!r}')

recipe = RECIPES[CELL]
USE_FOCK_V2 = recipe['use_fock_v2']
M_REGISTERS = recipe['n_registers']

print(f'Cell {CELL}: {recipe["desc"]}')
for k, v in recipe.items():
    print(f'  {k:30s} = {v!r}')

## 3. Load TinyStories (1M token subset)

In [ ]:
from data_module import load_tiny_stories, get_batch

train_ids, val_ids = load_tiny_stories(max_train_tokens=MAX_TRAIN_TOK)
print(f'train: {len(train_ids):,} tokens   val: {len(val_ids):,} tokens')

# Build logfreq surprisal for mass initialization
LOGFREQ_PATH = RESULTS_ROOT / 'logfreq_surprisal_tinystories.npy'
SCALEUP_LOGFREQ = SCALEUP_DIR / 'results' / 'logfreq_surprisal_tinystories.npy'
if SCALEUP_LOGFREQ.exists():
    LOGFREQ_PATH = SCALEUP_LOGFREQ
elif not LOGFREQ_PATH.exists():
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
    p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
    surprisal = (-np.log(p)).astype(np.float32)
    np.save(LOGFREQ_PATH, surprisal)
print(f'logfreq: {LOGFREQ_PATH}')

## 4. Build model

In [ ]:
import importlib
import torch.nn.functional as F_torch

import model_parf_sparse
import model_fock_parf_v2
importlib.reload(model_parf_sparse)
importlib.reload(model_fock_parf_v2)
from model_parf_sparse import SparsePARFConfig, SparsePARFLM
from model_fock_parf_v2 import FockPARFConfig_v2, FockPARFLM_v2

torch.manual_seed(SEED)

# Shared PARF config (P10g recipe)
parf_kwargs = dict(
    vocab_size=VOCAB_SIZE,
    d=D, max_len=MAX_LEN, L=L,
    v_hidden=V_HIDDEN, v_depth=V_DEPTH,
    v_phi_kind='structural_competitive',
    v_phi_d_type=16, v_phi_d_angle=8,
    v_phi_phi_hidden=64, v_phi_theta_hidden=64,
    v_phi_mlp_hidden=64,
    mass_mode='logfreq',
    logfreq_path=str(LOGFREQ_PATH),
    top_k=TOP_K,
    score_head_hidden=SCORE_HEAD,
    ln_before_distance=True,
    per_layer_v_phi_scale=True,
    theta_activation='softsign',
    theta_form='bilinear',
)

if USE_FOCK_V2:
    fock_kwargs = dict(
        n_registers=recipe['n_registers'],
        d_k=recipe['d_k'],
        register_salience_decay=recipe['register_salience_decay'],
        register_salience_threshold=recipe['register_salience_threshold'],
        destruction_gate_hidden=recipe['destruction_gate_hidden'],
        reverse_channel=recipe['reverse_channel'],
        stack_discipline=True,
    )
    if 'tau_create_init' in recipe:
        fock_kwargs['tau_create_init'] = recipe['tau_create_init']
    cfg = FockPARFConfig_v2(**parf_kwargs, **fock_kwargs)
    model = FockPARFLM_v2(cfg).to(device)
    n_fock = model.get_fock_v2_overhead()
else:
    cfg = SparsePARFConfig(**parf_kwargs)
    model = SparsePARFLM(cfg).to(device)
    n_fock = 0

n_total = sum(p.numel() for p in model.parameters())
n_v_theta = sum(p.numel() for p in model.V_theta.parameters())
print(f'Model: {type(model).__name__}')
print(f'params: total={n_total:,}  V_theta={n_v_theta:,}  Fock_overhead={n_fock:,}')

if USE_FOCK_V2:
    print(f'M={cfg.n_registers}  d_k={cfg.d_k}  reverse_channel={cfg.reverse_channel}')
    print(f'salience_decay={cfg.register_salience_decay}  '
          f'salience_threshold={cfg.register_salience_threshold}')
    print(f'reverse_channel_scale init = '
          f'{model.reverse_channel_scale.item():.4f} '
          f'→ tanh = {torch.tanh(model.reverse_channel_scale).item():.4f}')
    if model.creation_gate.log_tau is not None:
        tau_val = model.creation_gate.log_tau.exp().item()
        print(f'tau_create init = {tau_val:.4f}  '
              f'(vs 1/sqrt(d_k) = {1.0 / cfg.d_k**0.5:.4f})')
    else:
        print(f'tau_create = None (using 1/sqrt(d_k) = {1.0 / cfg.d_k**0.5:.4f})')

## 5. Monkey-patch for blend_scale ablation (D4)

If `blend_scale < 1.0`, we intercept the salience-weighted blending
in `_fock_v2_layer_step` to reduce register staleness.

In [ ]:
BLEND_SCALE = recipe.get('blend_scale', 1.0)

if USE_FOCK_V2 and BLEND_SCALE != 1.0:
    _original_fock_step = model._fock_v2_layer_step

    def _patched_fock_step(h, h_prev, r, salience, m_b, gamma, dt, layer_idx):
        # Scale down salience before blending to allow more fresh content
        salience_for_blend = salience * BLEND_SCALE
        # Temporarily replace salience, run the original, restore
        return _original_fock_step(
            h, h_prev, r, salience_for_blend, m_b, gamma, dt, layer_idx,
        )

    model._fock_v2_layer_step = _patched_fock_step
    print(f'Patched blend_scale = {BLEND_SCALE}')
else:
    print(f'blend_scale = {BLEND_SCALE} (no patch needed)')

## 6. Training loop

In [ ]:
def lr_at(step):
    if step < WARMUP_STEPS:
        return LR * (step + 1) / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(TOTAL_STEPS - WARMUP_STEPS, 1)
    return LR * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


GUMBEL_TAU_INIT = 1.0
GUMBEL_TAU_MIN  = 0.1

def tau_at(step):
    warm = int(0.2 * TOTAL_STEPS)
    if step < warm:
        return GUMBEL_TAU_INIT
    progress = (step - warm) / max(TOTAL_STEPS - warm, 1)
    return GUMBEL_TAU_INIT + (GUMBEL_TAU_MIN - GUMBEL_TAU_INIT) * min(progress, 1.0)


@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    for _ in range(EVAL_ITERS):
        xb, yb = get_batch(val_ids, BATCH_SIZE, BLOCK_SIZE, rng)
        x = torch.from_numpy(xb).to(device)
        y = torch.from_numpy(yb).to(device)
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


HAS_GUMBEL = hasattr(model, 'set_gumbel_tau')

opt = torch.optim.AdamW(
    model.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=WEIGHT_DECAY,
)
model.train()

log = []
best_ppl = float('inf')
t0 = time.time()

# Track reverse_channel_scale evolution (B5 diagnostic)
rc_scale_history = []
# Track tau_create evolution (B1 fix diagnostic)
tau_create_history = []
HAS_TAU_CREATE = (USE_FOCK_V2 and model.creation_gate.log_tau is not None)

for step in range(TOTAL_STEPS):
    for g in opt.param_groups:
        g['lr'] = lr_at(step)
    if HAS_GUMBEL:
        model.set_gumbel_tau(tau_at(step))

    xb, yb = get_batch(train_ids, BATCH_SIZE, BLOCK_SIZE, rng)
    x = torch.from_numpy(xb).to(device)
    y = torch.from_numpy(yb).to(device)

    _, loss = model(x, y)

    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    opt.step()

    # Track reverse channel scale
    if USE_FOCK_V2 and model.reverse_channel_scale is not None:
        with torch.no_grad():
            rc_val = model.reverse_channel_scale.item()
            rc_tanh = torch.tanh(model.reverse_channel_scale).item()
            rc_grad = (
                model.reverse_channel_scale.grad.item()
                if model.reverse_channel_scale.grad is not None else 0.0
            )
        if (step + 1) % LOG_EVERY == 0:
            rc_scale_history.append({
                'step': step + 1, 'raw': rc_val,
                'tanh': rc_tanh, 'grad': rc_grad,
            })

    # Track learnable creation temperature
    if HAS_TAU_CREATE:
        with torch.no_grad():
            tau_val = model.creation_gate.log_tau.exp().clamp(min=1e-4).item()
            tau_grad = (
                model.creation_gate.log_tau.grad.item()
                if model.creation_gate.log_tau.grad is not None else 0.0
            )
        if (step + 1) % LOG_EVERY == 0:
            tau_create_history.append({
                'step': step + 1, 'tau': tau_val, 'grad': tau_grad,
            })

    if (step + 1) % LOG_EVERY == 0 or step == 0:
        rc_str = ''
        if USE_FOCK_V2 and model.reverse_channel_scale is not None:
            rc_str = f'  rc_scale={rc_tanh:+.4f}'
        tau_str = ''
        if HAS_TAU_CREATE:
            tau_str = f'  tau={tau_val:.4f}'
        print(f'[{CELL}] step {step+1:>5}/{TOTAL_STEPS}  '
              f'lr={lr_at(step):.2e}  '
              f'train_loss={loss.item():.4f}{rc_str}{tau_str}  '
              f'wall={time.time()-t0:.0f}s')

    if (step + 1) % EVAL_EVERY == 0 or (step + 1) == TOTAL_STEPS:
        val_loss = evaluate()
        val_ppl = math.exp(val_loss)
        if val_ppl < best_ppl:
            best_ppl = val_ppl
        print(f'  >> val_loss={val_loss:.4f}  val_ppl={val_ppl:.2f}  '
              f'best_ppl={best_ppl:.2f}')
        log.append({
            'step': step + 1,
            'val_loss': val_loss,
            'val_ppl': val_ppl,
            'best_ppl': best_ppl,
            'train_loss': loss.item(),
        })

elapsed = time.time() - t0
print(f'\n[{CELL}] Done. wall={elapsed:.0f}s  '
      f'final_ppl={log[-1]["val_ppl"]:.2f}  best_ppl={best_ppl:.2f}')

## 7. Save checkpoint and log

In [ ]:
import dataclasses

full_tag = f'fock_v2_debug_{CELL}_seed{SEED}'
ckpt_path = RUN_DIR / f'{full_tag}_ckpt.pt'
log_path  = RUN_DIR / f'{full_tag}_training_log.jsonl'

torch.save({
    'model_state_dict': model.state_dict(),
    'config': dataclasses.asdict(cfg) if dataclasses.is_dataclass(cfg) else vars(cfg),
    'recipe': recipe,
    'best_ppl': best_ppl,
    'step': TOTAL_STEPS,
}, ckpt_path)

with open(log_path, 'w') as f:
    for entry in log:
        f.write(json.dumps(entry) + '\n')

if rc_scale_history:
    rc_path = RUN_DIR / f'{full_tag}_rc_scale_history.jsonl'
    with open(rc_path, 'w') as f:
        for entry in rc_scale_history:
            f.write(json.dumps(entry) + '\n')
    print(f'rc_scale history: {rc_path}')

if tau_create_history:
    tau_path = RUN_DIR / f'{full_tag}_tau_create_history.jsonl'
    with open(tau_path, 'w') as f:
        for entry in tau_create_history:
            f.write(json.dumps(entry) + '\n')
    print(f'tau_create history: {tau_path}')

print(f'checkpoint: {ckpt_path}')
print(f'log:        {log_path}')

## 8. Deep diagnostics — register mechanism instrumentation

This section instruments the Fock v2 forward pass to measure:
- **B1**: Register attention entropy and coverage per register per layer
- **B2**: Cross-layer creation-gate output similarity
- **B3**: Blend ratio and register content staleness per layer
- **B4**: Register-vs-token force magnitude in PARF dynamics
- **B5**: Reverse channel scale evolution and gradient signal

In [ ]:
import matplotlib.pyplot as plt

if not USE_FOCK_V2:
    print(f'Skipping register diagnostics for {CELL} (baseline arm).')
else:
    model.eval()
    diag_batches = 10

    # Accumulators
    all_attn_entropy = []       # (n_batches, L, M)
    all_alpha_max    = []       # (n_batches, L, M)
    all_salience     = []       # (n_batches, L, M)
    all_blend_ratio  = []       # (n_batches, L, M)  actual blend weight used
    all_content_cos  = []       # (n_batches, L, M)  cos(r_before, r_after)
    all_n_active     = []       # (n_batches, L)
    all_creation_out = []       # (n_batches, L, M, d) for cross-layer sim

    for b_idx in range(diag_batches):
        xb, _ = get_batch(val_ids, BATCH_SIZE, BLOCK_SIZE, rng)
        x = torch.from_numpy(xb).to(device)

        # PARF layer step uses torch.autograd.grad for force computation,
        # so we need enable_grad(). We detach after each layer to
        # prevent the graph from growing across the full stack.
        with torch.enable_grad():
            h0 = model._embed(x)
            r, salience = model._init_registers(BATCH_SIZE, h0.device)
            h, h_prev = h0, h0
            m_b = model.compute_mass(x)
            gamma, dt = model.gamma, cfg.dt

            batch_entropy = []
            batch_alpha_max = []
            batch_salience = []
            batch_blend = []
            batch_cos = []
            batch_active = []
            batch_creation = []

            for ell in range(cfg.L):
                B, T, d = h.shape
                M = cfg.n_registers

                # --- Manually instrument creation gate ---
                r_before = r.detach().clone()
                r_new_content, alpha_max_diag = model.creation_gate(h, r)

                # Attention entropy (B1): recompute attention weights
                Q_diag = torch.einsum("bmd,mdk->bmk", r, model.creation_gate.W_Q)
                K_diag = model.creation_gate.W_K(h)
                scores_diag = torch.bmm(
                    Q_diag.reshape(B * M, 1, cfg.d_k),
                    K_diag.unsqueeze(1).expand(B, M, T, cfg.d_k)
                         .reshape(B * M, cfg.d_k, T),
                ).reshape(B, M, T)
                if model.creation_gate.log_tau is not None:
                    tau_diag = model.creation_gate.log_tau.exp().clamp(min=1e-4)
                    scores_diag = scores_diag / tau_diag
                else:
                    scores_diag = scores_diag / (cfg.d_k ** 0.5)
                alpha_diag = F_torch.softmax(scores_diag, dim=-1)  # (B, M, T)

                # Entropy: -Σ p log p
                log_alpha = torch.log(alpha_diag + 1e-12)
                entropy = -(alpha_diag * log_alpha).sum(dim=-1)  # (B, M)
                max_entropy = math.log(T)
                norm_entropy = entropy / max_entropy  # 0=peaked, 1=uniform

                batch_entropy.append(norm_entropy.detach().mean(0).cpu().numpy())
                batch_alpha_max.append(alpha_max_diag.detach().mean(0).cpu().numpy())

                # Blending (B3)
                blend = salience.unsqueeze(-1)
                r_blended = blend * r + (1.0 - blend) * r_new_content

                # Content cosine similarity before vs after creation
                cos_sim = F_torch.cosine_similarity(
                    r_before.reshape(-1, d),
                    r_blended.detach().reshape(-1, d), dim=-1,
                ).reshape(B, M)
                batch_cos.append(cos_sim.detach().mean(0).cpu().numpy())
                batch_blend.append(salience.detach().mean(0).cpu().numpy())

                # Store creation output for cross-layer similarity
                batch_creation.append(r_new_content.detach().mean(0).cpu().numpy())  # (M, d)

                # Run the actual layer step
                h_new, h_prev_out, r, salience = model._fock_v2_layer_step(
                    h, h_prev, r, salience, m_b, gamma, dt, layer_idx=ell,
                )
                # Detach to prevent graph from growing across layers
                h_new = h_new.detach().requires_grad_(True)
                h_prev_out = h_prev_out.detach().requires_grad_(True)
                r = r.detach()
                salience = salience.detach()

                h_prev = h_prev_out
                h = h_new

                active = model._active_mask(salience)
                batch_active.append(active.sum(-1).float().mean().item())
                batch_salience.append(salience.detach().mean(0).cpu().numpy())

            all_attn_entropy.append(np.stack(batch_entropy))    # (L, M)
            all_alpha_max.append(np.stack(batch_alpha_max))     # (L, M)
            all_salience.append(np.stack(batch_salience))       # (L, M)
            all_blend_ratio.append(np.stack(batch_blend))       # (L, M)
            all_content_cos.append(np.stack(batch_cos))         # (L, M)
            all_n_active.append(batch_active)                   # (L,)
            all_creation_out.append(np.stack(batch_creation))   # (L, M, d)

    # Average over batches
    mean_entropy   = np.mean(all_attn_entropy, axis=0)    # (L, M)
    mean_alpha_max = np.mean(all_alpha_max, axis=0)       # (L, M)
    mean_salience  = np.mean(all_salience, axis=0)        # (L, M)
    mean_blend     = np.mean(all_blend_ratio, axis=0)     # (L, M)
    mean_cos       = np.mean(all_content_cos, axis=0)     # (L, M)
    mean_active    = np.mean(all_n_active, axis=0)        # (L,)
    mean_creation  = np.mean(all_creation_out, axis=0)    # (L, M, d)

    print(f'Diagnostics over {diag_batches} batches:')
    print(f'  Mean active registers per layer: {mean_active.tolist()}')
    print(f'  Mean attention entropy (normalised, 0=peaked 1=uniform):')
    for ell in range(cfg.L):
        print(f'    Layer {ell}: {mean_entropy[ell].mean():.3f}  '
              f'(range: {mean_entropy[ell].min():.3f}..{mean_entropy[ell].max():.3f})')
    print(f'  Mean content cosine (1.0 = fully stale, no new info):')
    for ell in range(cfg.L):
        print(f'    Layer {ell}: {mean_cos[ell].mean():.3f}')

    # ── Figure 1: 4-panel overview ──
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # (a) Attention entropy heatmap (B1)
    im0 = axes[0, 0].imshow(mean_entropy, aspect='auto', cmap='RdYlGn_r')
    axes[0, 0].set_xlabel('Register index')
    axes[0, 0].set_ylabel('Layer')
    axes[0, 0].set_title('B1: Normalised attention entropy\n(0=peaked → useful, 1=uniform → wasted)')
    plt.colorbar(im0, ax=axes[0, 0])

    # (b) Salience heatmap
    im1 = axes[0, 1].imshow(mean_salience, aspect='auto', cmap='viridis')
    axes[0, 1].set_xlabel('Register index')
    axes[0, 1].set_ylabel('Layer')
    axes[0, 1].set_title('Salience per register per layer\n(after layer step)')
    plt.colorbar(im1, ax=axes[0, 1])

    # (c) Content cosine similarity (B3: staleness)
    im2 = axes[1, 0].imshow(mean_cos, aspect='auto', cmap='Reds', vmin=0.5, vmax=1.0)
    axes[1, 0].set_xlabel('Register index')
    axes[1, 0].set_ylabel('Layer')
    axes[1, 0].set_title('B3: Content cosine(before, after creation)\n(1.0 = fully stale)')
    plt.colorbar(im2, ax=axes[1, 0])

    # (d) Active register count per layer
    axes[1, 1].bar(range(cfg.L), mean_active)
    axes[1, 1].set_xlabel('Layer')
    axes[1, 1].set_ylabel('Mean active registers')
    axes[1, 1].set_title(f'Active registers per layer (max={M_REGISTERS})')
    axes[1, 1].set_xticks(range(cfg.L))

    plt.suptitle(f'{CELL}: {recipe["desc"]}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    fig.savefig(RUN_DIR / f'{full_tag}_register_diagnostics.png', dpi=150)
    plt.show()

    # ── Figure 2: Cross-layer creation similarity (B2) ──
    creation_flat = mean_creation.reshape(cfg.L, -1)  # (L, M*d)
    cross_layer_sim = np.corrcoef(creation_flat)      # (L, L)

    fig2, ax2 = plt.subplots(figsize=(6, 5))
    im3 = ax2.imshow(cross_layer_sim, cmap='coolwarm', vmin=-1, vmax=1)
    ax2.set_xlabel('Layer')
    ax2.set_ylabel('Layer')
    ax2.set_title('B2: Cross-layer creation output correlation\n'
                  '(high everywhere = shared gate is redundant)')
    ax2.set_xticks(range(cfg.L))
    ax2.set_yticks(range(cfg.L))
    plt.colorbar(im3, ax=ax2)
    plt.tight_layout()
    fig2.savefig(RUN_DIR / f'{full_tag}_cross_layer_creation_sim.png', dpi=150)
    plt.show()

    # ── Figure 3: Reverse channel scale evolution (B5) ──
    if rc_scale_history:
        fig3, axes3 = plt.subplots(1, 2, figsize=(12, 4))
        rc_steps = [r['step'] for r in rc_scale_history]
        rc_tanhs = [r['tanh'] for r in rc_scale_history]
        rc_grads = [r['grad'] for r in rc_scale_history]

        axes3[0].plot(rc_steps, rc_tanhs, 'b-')
        axes3[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
        axes3[0].set_xlabel('Step')
        axes3[0].set_ylabel('tanh(reverse_channel_scale)')
        axes3[0].set_title('B5: Reverse channel gate value\n'
                           '(0 = off, ±1 = fully open)')
        axes3[0].grid(True, alpha=0.3)

        axes3[1].plot(rc_steps, rc_grads, 'r-', alpha=0.7)
        axes3[1].axhline(0, color='gray', linestyle='--', alpha=0.5)
        axes3[1].set_xlabel('Step')
        axes3[1].set_ylabel('Gradient of reverse_channel_scale')
        axes3[1].set_title('B5: Gradient signal for reverse channel')
        axes3[1].grid(True, alpha=0.3)

        plt.suptitle(f'{CELL}: Reverse channel evolution', fontweight='bold')
        plt.tight_layout()
        fig3.savefig(RUN_DIR / f'{full_tag}_reverse_channel.png', dpi=150)
        plt.show()

        print(f'\nReverse channel scale at end of training:')
        print(f'  raw = {rc_scale_history[-1]["raw"]:.4f}')
        print(f'  tanh = {rc_scale_history[-1]["tanh"]:.4f}')
    else:
        print('No reverse channel history (channel disabled or baseline arm).')

    # ── Figure 4: tau_create evolution (B1 fix) ──
    if tau_create_history:
        fig4, axes4 = plt.subplots(1, 2, figsize=(12, 4))
        tc_steps = [r['step'] for r in tau_create_history]
        tc_taus  = [r['tau'] for r in tau_create_history]
        tc_grads = [r['grad'] for r in tau_create_history]

        axes4[0].plot(tc_steps, tc_taus, 'b-', linewidth=2)
        axes4[0].axhline(1.0 / cfg.d_k**0.5, color='orange', linestyle='--',
                         alpha=0.6, label=f'1/√d_k = {1.0/cfg.d_k**0.5:.3f}')
        axes4[0].set_xlabel('Step')
        axes4[0].set_ylabel('τ_create')
        axes4[0].set_title('B1 fix: Learnable creation temperature\n'
                           '(low → peaked, high → uniform)')
        axes4[0].legend()
        axes4[0].grid(True, alpha=0.3)

        axes4[1].plot(tc_steps, tc_grads, 'r-', alpha=0.7)
        axes4[1].axhline(0, color='gray', linestyle='--', alpha=0.5)
        axes4[1].set_xlabel('Step')
        axes4[1].set_ylabel('∂L/∂log(τ)')
        axes4[1].set_title('Gradient signal for log(τ_create)')
        axes4[1].grid(True, alpha=0.3)

        plt.suptitle(f'{CELL}: Creation temperature evolution', fontweight='bold')
        plt.tight_layout()
        fig4.savefig(RUN_DIR / f'{full_tag}_tau_create.png', dpi=150)
        plt.show()

        print(f'\nCreation temperature at end of training:')
        print(f'  tau = {tau_create_history[-1]["tau"]:.6f}')
        print(f'  1/sqrt(d_k) reference = {1.0 / cfg.d_k**0.5:.4f}')

    # Save diagnostic stats as JSON
    diag_stats = {
        'mean_active_per_layer': mean_active.tolist(),
        'mean_entropy_per_layer': [float(mean_entropy[ell].mean())
                                   for ell in range(cfg.L)],
        'mean_content_cosine_per_layer': [float(mean_cos[ell].mean())
                                          for ell in range(cfg.L)],
        'cross_layer_creation_corr': cross_layer_sim.tolist(),
    }
    if rc_scale_history:
        diag_stats['final_rc_scale_tanh'] = rc_scale_history[-1]['tanh']
    if tau_create_history:
        diag_stats['final_tau_create'] = tau_create_history[-1]['tau']
    with open(RUN_DIR / f'{full_tag}_diag_stats.json', 'w') as f:
        json.dump(diag_stats, f, indent=2)
    print(f'\nDiagnostic stats saved to {full_tag}_diag_stats.json')

## 9. Training curves

In [ ]:
import matplotlib.pyplot as plt

steps_arr = [e['step'] for e in log]
ppls = [e['val_ppl'] for e in log]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(steps_arr, ppls, 'o-', label=f'{CELL}: {recipe["desc"]}', linewidth=2)
ax.axhline(26.42, color='orange', linestyle='--', alpha=0.6,
           label='PARFLM P10g (26.42, v0 ceiling)')
ax.axhline(20.0, color='green', linestyle='--', alpha=0.6,
           label='Target (≤20 PPL)')
ax.axhline(7.81, color='red', linestyle=':', alpha=0.6,
           label='Matched-attn (7.81)')
ax.set_xlabel('Training step')
ax.set_ylabel('Val PPL')
ax.set_title(f'{CELL}: {recipe["desc"]}\n'
             f'best={best_ppl:.2f}  final={ppls[-1]:.2f}')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(RUN_DIR / f'{full_tag}_val_ppl.png', dpi=120)
plt.show()

## 10. Cross-cell comparison dashboard

After running all 5 cells (D1–D5), this collects results and shows
which bottleneck intervention actually moved the needle.

In [ ]:
results = {}
for cell_name in ('D1', 'D2', 'D3', 'D4', 'D5', 'D6'):
    cell_dir = RESULTS_ROOT / cell_name / f'seed{SEED}'
    if not cell_dir.exists():
        results[cell_name] = None
        continue
    logs = sorted(cell_dir.glob('*_training_log.jsonl'))
    if not logs:
        results[cell_name] = None
        continue
    rows = [json.loads(line) for line in logs[-1].read_text().splitlines()]
    best = min(r['val_ppl'] for r in rows) if rows else None
    final = rows[-1]['val_ppl'] if rows else None

    # Load diagnostic stats if available
    diag_files = sorted(cell_dir.glob('*_diag_stats.json'))
    diag = None
    if diag_files:
        diag = json.loads(diag_files[-1].read_text())

    results[cell_name] = {
        'desc': RECIPES[cell_name]['desc'],
        'best_ppl': best, 'final_ppl': final,
        'diag': diag,
    }

print(f'{"Cell":<6} {"Description":<55} {"best PPL":>10} {"final PPL":>10} '
      f'{"rc_scale":>10} {"entropy_L0":>10} {"tau_create":>10}')
print('-' * 130)

for cell_name in ('D1', 'D2', 'D3', 'D4', 'D5', 'D6'):
    r = results[cell_name]
    if r is None:
        desc = RECIPES.get(cell_name, {}).get('desc', '?')
        print(f'{cell_name:<6} {desc:<55} {"—":>10} {"—":>10} {"—":>10} {"—":>10} {"—":>10}')
        continue
    desc = r['desc']
    bp = f"{r['best_ppl']:.1f}" if r['best_ppl'] else '—'
    fp = f"{r['final_ppl']:.1f}" if r['final_ppl'] else '—'
    rc = '—'
    ent = '—'
    tc = '—'
    if r['diag']:
        rc_val = r['diag'].get('final_rc_scale_tanh')
        rc = f"{rc_val:.3f}" if rc_val is not None else '—'
        ent_vals = r['diag'].get('mean_entropy_per_layer')
        ent = f"{ent_vals[0]:.3f}" if ent_vals else '—'
        tc_val = r['diag'].get('final_tau_create')
        tc = f"{tc_val:.4f}" if tc_val is not None else '—'
    print(f'{cell_name:<6} {desc:<55} {bp:>10} {fp:>10} {rc:>10} {ent:>10} {tc:>10}')

print(f'\nReference: PARFLM P10g = 26.42 PPL  |  Matched-attn = 7.81 PPL')
print('\nInterpretation guide:')
print('  D2 >> D1 → B1 (register count) is binding')
print('  D3 >> D1 → B2 (shared gate) is binding')
print('  D4 >> D1 → B3 (salience staleness) is binding')
print('  D5 ≈  D1 → B6 (V_theta ceiling) dominates; registers provide no lift')
print('  D1 >> D5 → Registers help; focus on which sub-bottleneck (B1–B5) limits them')
print('  D6 >> D1 → B1 fix (learnable τ) sharpens creation attention; check entropy drop')

## 11. Verdict and next steps

**Decision rules after running all 5 cells:**

| Outcome | Diagnosis | Action |
|---|---|---|
| D1 ≈ D5 (registers provide no PPL lift) | B6: V_theta ceiling dominates | Register mechanism is irrelevant for next-token prediction at this scale; focus on V_theta depth/width |
| D2 >> D1 (more registers helps) | B1: Register compression | Scale M proportionally to T (e.g. M=64 for T=512) |
| D3 >> D1 (per-layer gates help) | B2: Shared gate redundancy | Implement per-layer W_K/W_V (cost: L× more params in creation gate) |
| D4 >> D1 (lower blend helps) | B3: Salience staleness | Reduce decay, lower persistence, or make blend a learnable per-layer parameter |
| Entropy near 1.0 at all layers | B1 variant: attention too diffuse | Reduce d_k or add temperature scaling to sharpen creation gate |
| D6 >> D1 (learnable τ helps) | B1 fix validated | Learnable temperature sharpens creation attention; consider combining with D3 (wider queries) |
| rc_scale stays ≈ 0 | B5: Reverse channel not learning | Increase initial scale, add auxiliary loss on register readout, or warm-start |

**Expected wall-clock:** ~20-40 min per arm on T4, ~10-15 min per arm on A100.
Total for all 5 arms: ~2-3 hours on T4, ~1 hour on A100.